# Phase 4 NOAA Weather Scenarios

This notebook validates weather-state probabilities and friction parameters generated for the connectivity engine.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid')

ROOT = Path.cwd()
if not (ROOT / 'data').exists() and (ROOT.parent / 'data').exists():
    ROOT = ROOT.parent

prob_path = ROOT / 'data' / 'processed' / 'weather_probabilities.csv'
friction_path = ROOT / 'src' / 'weather' / 'friction_params.json'

weather_probs = pd.read_csv(prob_path)
monthly = weather_probs[weather_probs['season_or_month'].str.startswith('month_')].copy()
monthly['month'] = monthly['season_or_month'].str.replace('month_', '', regex=False).astype(int)
monthly = monthly.sort_values(['month', 'weather_state'])

print('Probability rows:', len(weather_probs))
print('Unique groups:', weather_probs['season_or_month'].nunique())
weather_probs.head()

In [ ]:
meta_path = ROOT / 'data' / 'processed' / 'weather_scenario_metadata.json'
classified_path = ROOT / 'data' / 'processed' / 'weather_daily_states.csv'

with open(meta_path, 'r', encoding='utf-8') as f:
    weather_meta = json.load(f)

group_sums = weather_probs.groupby('season_or_month', as_index=False)['probability'].sum()
max_abs_deviation = (group_sums['probability'] - 1.0).abs().max()

print('Daily classified rows:', len(pd.read_csv(classified_path)))
print('Rule precedence:', ' > '.join(weather_meta['rule_precedence']))
print('Max |sum(prob)-1| across groups:', max_abs_deviation)

pd.DataFrame([
    {
        'date_min': weather_meta['date_min'],
        'date_max': weather_meta['date_max'],
        'rows_after_normalization': weather_meta['rows_after_normalization'],
        'heavy_rain_inches': weather_meta['thresholds']['heavy_rain_inches'],
        'cold_tmax_f': weather_meta['thresholds']['cold_tmax_f'],
        'heat_tmax_f': weather_meta['thresholds']['heat_tmax_f'],
        'windy_awnd_mph': weather_meta['thresholds']['windy_awnd_mph'],
    }
])

In [ ]:
state_order = ['Dry / Normal', 'Light Rain', 'Heavy Rain', 'Cold/Windy', 'Heat']
month_pivot = monthly.pivot(index='month', columns='weather_state', values='probability').fillna(0)
month_pivot = month_pivot[state_order]

fig, ax = plt.subplots(figsize=(12, 6))
month_pivot.plot(kind='bar', stacked=True, ax=ax)
ax.set_title('Weather State Distribution by Month')
ax.set_xlabel('Month')
ax.set_ylabel('Probability')
ax.legend(title='Weather State', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
summary_groups = ['Winter', 'Spring', 'Summer', 'Fall', 'Annual']
summary = weather_probs[weather_probs['season_or_month'].isin(summary_groups)].copy()
summary_pivot = summary.pivot(index='season_or_month', columns='weather_state', values='probability').fillna(0)
summary_pivot = summary_pivot.reindex(summary_groups)[state_order]

fig, ax = plt.subplots(figsize=(10, 5))
summary_pivot.plot(kind='bar', stacked=True, ax=ax)
ax.set_title('Seasonal and Annual Weather Profiles')
ax.set_xlabel('Season/Year')
ax.set_ylabel('Probability')
ax.legend(title='Weather State', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
with open(friction_path, 'r', encoding='utf-8') as f:
    friction = json.load(f)

friction_df = pd.DataFrame(friction).T[[
    'walk_speed_multiplier',
    'max_walk_distance_m',
    'transfer_penalty_mins',
]]
friction_df